<a href="https://colab.research.google.com/github/8lmer555/Stem-summer-program/blob/main/Team_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Fetch all bike networks from CityBikes API
import requests
url = "https://api.citybik.es/v2/networks"
response = requests.get(url)
# print("Status code:", response.status_code)
data = response.json()
print("Keys in response:", data.keys())
print("Number of networks:", len(data['networks']))
print("First network:", data['networks'][0])

#Filter to find NYC networks only
for item in data['networks']:
    if 'New York' in item['location']['city']:
        print(item)
url = "https://api.citybik.es/v2/networks/citi-bike-nyc"
result = requests.get(url)
print("Status code:", result.status_code)
internal_data = result.json()
print("Keys in results:",internal_data.keys())
print("Number of stations:", len(internal_data['network']['stations']))
print("First station:", internal_data['network']['stations'][0])

# Store stations in a Pandas DataFrame
import pandas as pd
stations = internal_data['network']['stations']
df = pd.DataFrame(stations)
df.head(10)
print(df.shape)
print(df.isnull().sum())

# Clean the data and remove unnecessary columns
df = df.drop(columns=['extra', 'timestamp', 'id'])
print(df.duplicated().sum())
df.head(5)

Keys in response: dict_keys(['networks'])
Number of networks: 782
First network: {'id': 'abu-dhabi-careem-bike', 'name': 'Abu Dhabi Careem BIKE', 'location': {'latitude': 24.4866, 'longitude': 54.3728, 'city': 'Abu Dhabi', 'country': 'AE'}, 'href': '/v2/networks/abu-dhabi-careem-bike', 'company': ['Careem'], 'gbfs_href': 'https://dubai.publicbikesystem.net/customer/gbfs/v2/en/gbfs.json'}
{'id': 'citi-bike-nyc', 'name': 'Citi Bike', 'location': {'latitude': 40.7143528, 'longitude': -74.00597309999999, 'city': 'New York, NY', 'country': 'US'}, 'href': '/v2/networks/citi-bike-nyc', 'company': ['NYC Bike Share, LLC', 'Motivate International, Inc.', 'PBSC Urban Solutions'], 'ebikes': True, 'gbfs_href': 'https://gbfs.citibikenyc.com/gbfs/gbfs.json'}
{'id': 'joco-new-york', 'name': 'JOCO New York', 'location': {'latitude': 40.7572, 'longitude': -73.9859, 'city': 'New York', 'country': 'US'}, 'href': '/v2/networks/joco-new-york', 'company': ['BEAM BIKE CORP.', 'JOCO'], 'ebikes': True}
Status c

,name,latitude,longitude,free_bikes,empty_slots
0,Chester Ave & 12 Ave,40.644367,-73.984276,13,7
1,W 56 St & 8 Ave,40.765959,-73.983135,3,40
2,S Portland Ave & Hanson Pl,40.685396,-73.974315,23,0
3,E 78 St & 2 Ave,40.772797,-73.955778,19,4
4,87 St & Roosevelt Ave,40.748250,-73.879360,2,21


In [ ]:
# print(df.duplicated().sum())
# print(df.shape)

In [ ]:
#Fetch all subway stops from Transistland API
import requests
API_KEY = "9oeaHhv5e0jFlPTYM0ZkaLY6xIoUr9sX"
url = "https://transit.land/api/v2/rest/stops"
headers = {"apikey": API_KEY}
params = {
    "feed_onestop_id": "f-dr5r-nyctsubway",
    "limit": 400
}
response = requests.get(url, headers=headers, params=params)
data = response.json()
print("Status code:", response.status_code)
print("Keys:", data.keys())
print("Number of stops:", len(data['stops']))
print("First stop:", data['stops'][0])
subway_stops = data['stops']

#Filter to find NYC subway stops only
subway_list = []
for stop in subway_stops:
    subway_list.append({
        'name': stop['stop_name'],
        'latitude': stop['geometry']['coordinates'][1],
        'longitude': stop['geometry']['coordinates'][0]
    })
#Store subway stops in Panda Dataframe
df_subway = pd.DataFrame(subway_list)
df_subway.head(5)

Status code: 200
Keys: dict_keys(['meta', 'stops'])
Number of stops: 400
First stop: {'feed_version': {'feed': {'id': 5, 'onestop_id': 'f-dr5r-nyctsubway'}, 'fetched_at': '2026-02-27T03:47:47.02807Z', 'id': 644578, 'sha1': 'f51d0e0444d677217e631b0fd1744414c2185207'}, 'geometry': {'coordinates': [-73.898583, 40.889248], 'type': 'Point'}, 'id': 2065499828, 'level': None, 'location_type': 1, 'onestop_id': 's-dr72w7stnh-vancortlandtpark~242st', 'parent': None, 'place': {'adm0_iso': 'US', 'adm0_name': 'United States of America', 'adm1_iso': 'US-NY', 'adm1_name': 'New York'}, 'platform_code': None, 'stop_code': None, 'stop_desc': None, 'stop_id': '101', 'stop_name': 'Van Cortlandt Park-242 St', 'stop_timezone': None, 'stop_url': None, 'tts_stop_name': None, 'wheelchair_boarding': None, 'zone_id': None}


,name,latitude,longitude
0,Van Cortlandt Park-242 St,40.889248,-73.898583
1,238 St,40.884667,-73.900870
2,231 St,40.878856,-73.904834
3,Marble Hill-225 St,40.874561,-73.909831
4,215 St,40.869444,-73.915279


In [ ]:
#Checking if there is any possible errors
# print(df_subway.isnull().sum())
# print(df_subway.duplicated().sum())

In [ ]:
from geopy.distance import geodesic
results = []

for i, citi_row in df.iterrows():
    citi_location = (citi_row['latitude'], citi_row['longitude'])
    nearest_distance = float('inf')
    nearest_stop = ""

    for j, subway_row in df_subway.iterrows():
        subway_location = (subway_row['latitude'], subway_row['longitude'])
        distance = geodesic(citi_location, subway_location).kilometers

        if distance < nearest_distance:
            nearest_distance = distance
            nearest_stop = subway_row['name']

    results.append({
        'citi_station': citi_row['name'],
        'nearest_subway': nearest_stop,
        'distance_km': round(nearest_distance, 2)
    })

df_results = pd.DataFrame(results)
df_results.head(10)
print(df_results.shape)
print(df_results['distance_km'].describe())

(2317, 3)
count    2317.000000
mean        0.733185
std         0.951018
min         0.000000
25%         0.240000
50%         0.400000
75%         0.760000
max         7.100000
Name: distance_km, dtype: float64


In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df_results['distance_km'], bins=30, color='steelblue', edgecolor='black')
plt.title('Distance from Citi Bike Stations to Nearest Subway Stop')
plt.xlabel('Distance (km)')
plt.ylabel('Number of Stations')
plt.axvline(df_results['distance_km'].mean(), color='red', linestyle='--', label=f'Mean: {df_results["distance_km"].mean():.2f} km')
plt.legend()
plt.show()

NameError: name 'plt' is not defined

In [ ]:
far_stations = df_results[df_results['distance_km'] > 1.5]
print("Number of underserved stations:", len(far_stations))

In [ ]:
dependent_areas = far_stations['nearest_subway'].value_counts().head(10)
dependent_areas.plot(kind='bar', figsize=(10,5), color='tomato')
plt.title("Subway Areas with Most Underserved Citi Bike Stations")
plt.xlabel("Nearest Subway Station")
plt.ylabel("Number of Far Citi Bike Stations")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
df['total_capacity'] = df['empty_slots'] + df['free_bikes']
df['pct_available'] = df['free_bikes']/df['total_capacity'] * 100
#combined df and df_results since data we want is on both
df_combined = df.merge(df_results, left_on='name', right_on='citi_station')
df_filtered = df_combined[(df_combined['distance_km'] < 1.5) & (df_combined['pct_available'] < 20)]
df_filtered = df_filtered.sort_values('pct_available',ascending = True )
print(df_filtered[['name', 'distance_km', 'pct_available', 'free_bikes']].head(10))
print(len(df_filtered))



In [ ]:
plt.scatter(df_filtered['distance_km'], df_filtered['pct_available'])
plt.xlabel('Distance from Subway (km)')
plt.ylabel('Bike Availability (%)')
plt.title('Bike Availability vs Distance from Subway Stations')
plt.show()

In [ ]:
top10 = df_filtered.head(10)
top10.plot(kind='bar', x='name', y='pct_available')
plt.xlabel('station_name')
plt.ylabel('percentage %')
plt.title('Top 10 struggling stations ')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show


In [ ]:
print(df_results.columns.tolist())
print(df_results.head(5))